# Phase 4: Category Models and Thresholds
English training data: 500k natural comments (learning curve: natural > enriched for
toxicity) plus all comments from rare categories (needed for threat etc.).

In [ ]:
%load_ext autoreload
%autoreload 2

import joblib
import pandas as pd

from portfolio_projekt.data import DATA_DIR, LABELS, MODELS_DIR, THRESHOLD, load_english, load_german
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score

In [ ]:
# Validation and test stay unchanged; German data is used as prepared in Phase 2
_, df_en_val, df_en_test = load_english()
df_de_train, df_de_val, df_de_test = load_german()

# Full English training split without texts that appear in validation or test
df_en_full = pd.read_parquet(DATA_DIR / "raw" / "civil_comments" / "train")
in_eval = df_en_full["text"].isin(df_en_val["text"]) | df_en_full["text"].isin(df_en_test["text"])
df_en_full = df_en_full[~in_eval].drop_duplicates(subset="text")

# 500k natural comments plus all comments from rare categories
rare_labels = ["obscene", "threat", "identity_attack", "sexual_explicit"]
is_rare = (df_en_full[rare_labels] >= THRESHOLD).any(axis=1)

df_en_train = pd.concat([
    df_en_full.sample(500_000, random_state=42),
    df_en_full[is_rare],
]).drop_duplicates(subset="text")

del df_en_full
print(df_en_train.shape)
(df_en_train[LABELS] >= THRESHOLD).mean().mul(100).round(2)

## English category models
One shared word TF-IDF (unigrams + bigrams), one logistic regression per label.

In [ ]:
def evaluate(y_true, p, name):
    """Return ROC-AUC, PR-AUC and the best F1 score with its threshold."""
    precision, recall, thresholds = precision_recall_curve(y_true, p)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    best = f1[:-1].argmax()
    return {
        "label": name,
        "roc_auc": roc_auc_score(y_true, p),
        "pr_auc": average_precision_score(y_true, p),
        "best_threshold": thresholds[best],
        "best_f1": f1[best],
    }

In [ ]:
vectorizer_en = TfidfVectorizer(ngram_range=(1, 2), min_df=20, max_features=300_000, sublinear_tf=True)
X_en_train = vectorizer_en.fit_transform(df_en_train["text"])
X_en_val = vectorizer_en.transform(df_en_val["text"])
print(X_en_train.shape)

In [ ]:
models_en = {}
results_en = []

for label in LABELS:
    y_train = df_en_train[label] >= THRESHOLD
    y_val = df_en_val[label] >= THRESHOLD

    model = LogisticRegression(max_iter=1000)
    model.fit(X_en_train, y_train)
    p_val = model.predict_proba(X_en_val)[:, 1]

    models_en[label] = model
    result = evaluate(y_val, p_val, label)
    result["base_rate"] = y_val.mean()
    result["positives_val"] = int(y_val.sum())
    results_en.append(result)
    print(f"{label:16} PR-AUC {result['pr_auc']:.3f}")

df_results_en = pd.DataFrame(results_en)
df_results_en["pr_auc_vs_base"] = df_results_en["pr_auc"] / df_results_en["base_rate"]
df_results_en.round(3)

In [ ]:
for label in LABELS:
    y_train = df_en_train[label] >= THRESHOLD
    p_train = models_en[label].predict_proba(X_en_train)[:, 1]
    auc_train = roc_auc_score(y_train, p_train)
    auc_val = df_results_en.set_index("label").loc[label, "roc_auc"]
    print(f"{label:16} train {auc_train:.3f}   val {auc_val:.3f}   gap {auc_train - auc_val:+.3f}")

**Findings (English category models):** ROC-AUC is high for all labels (0.94–0.97), but
PR-AUC reveals large differences: toxicity and insult ~0.68, obscene 0.51, identity_attack
0.30, sexual_explicit 0.21, threat 0.18. For rare classes ROC-AUC is misleadingly optimistic.
threat, the label that triggers the highest recommendation level, is the weakest
→ in recommend(), a threat signal should lead to "manual review" rather than a hard action.
Only ~200 validation examples for threat and sexual_explicit → uncertain estimates.

**Overfitting check:** ROC-AUC gap between training and validation is small for all labels
(+0.007 to +0.022), so there is no meaningful overfitting. Training ROC-AUC itself stays
at 0.95–0.98: a linear word model is too simple for meaning, sarcasm and implicit threats
(underfitting in terms of model capacity, as the error analysis showed).

## German category models
Category labels exist only for RP, so
category models are trained on RP rows only. threat is skipped (too few examples).

In [ ]:
print(f"German training threats: {(df_de_train['threat'] >= THRESHOLD).sum()}")

In [ ]:
DE_LABELS = ["toxicity", "insult", "obscene", "identity_attack"]

vectorizer_de = TfidfVectorizer(ngram_range=(1, 2), min_df=20, max_features=300_000, sublinear_tf=True)
X_de_train = vectorizer_de.fit_transform(df_de_train["text"])
X_de_val = vectorizer_de.transform(df_de_val["text"])

models_de = {}
results_de = []

for label in DE_LABELS:
    # Use only rows where this label exists (categories are NaN for GermEval)
    has_label_train = df_de_train[label].notna().values
    has_label_val = df_de_val[label].notna().values

    y_train = df_de_train.loc[has_label_train, label] >= THRESHOLD
    y_val = df_de_val.loc[has_label_val, label] >= THRESHOLD

    model = LogisticRegression(max_iter=1000)
    model.fit(X_de_train[has_label_train], y_train)
    p_val = model.predict_proba(X_de_val[has_label_val])[:, 1]

    models_de[label] = model
    result = evaluate(y_val, p_val, label)
    result["base_rate"] = y_val.mean()
    result["positives_val"] = int(y_val.sum())
    results_de.append(result)

df_results_de = pd.DataFrame(results_de)
df_results_de["pr_auc_vs_base"] = df_results_de["pr_auc"] / df_results_de["base_rate"]
df_results_de.round(3)

**Decision: continue with English only.** German category models are barely usable
(insult 0.087, obscene 0.040 with only 28 validation examples, identity_attack 0.140),
only 22 German threats exist, and without character n-grams German toxicity drops from
PR-AUC 0.250 to 0.178. German needs more and more consistent data and a subword-aware,
multilingual model (see README: "Why English only?"). The German cells above remain as
evidence for this decision.

## Saving the English model bundle
Vectorizer, one model per label and the best-F1 thresholds from the validation data.

In [ ]:
thresholds_en = df_results_en.set_index("label")["best_threshold"].round(3).to_dict()

bundle_en = {
    "vectorizer": vectorizer_en,
    "models": models_en,
    "thresholds": thresholds_en,
    "labels": LABELS,
}

MODELS_DIR.mkdir(exist_ok=True)
joblib.dump(bundle_en, MODELS_DIR / "tfidf_en.joblib")
print(thresholds_en)

In [ ]:
bundle = joblib.load(MODELS_DIR / "tfidf_en.joblib")
X_example = bundle["vectorizer"].transform(["You are an idiot", "Thanks for the article"])

{label: model.predict_proba(X_example)[:, 1].round(2).tolist() for label, model in bundle["models"].items()}